# Streaming SSE Server — LangGraph with astream_events
## Push Tokens as They Arrive — SSE and astream_events
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/81-streaming-sse-server/streaming_sse_server_workbook.ipynb)

Demonstrates a think→answer LangGraph graph with token-level streaming via astream_events. Shows the FastAPI + EventSourceResponse pattern as reference code. The streaming graph itself runs in Colab.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — SSE vs polling; when streaming UX matters |
| 2 | **Streaming Graph** — think_node → answer_node; both nodes use streaming ChatOpenAI |
| 3 | **astream_events v1** — Filter on_chat_model_stream to print tokens as they arrive |
| 4 | **Batch vs Stream** — Compare output latency: invoke vs astream_events |
| 5 | **FastAPI Reference** — Full SSE server pattern (reference only) |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
import asyncio, time
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# two-node graph: think first, then answer — streaming shows tokens from BOTH nodes as they arrive
class ThinkAnswerState(TypedDict):
    question: str; answer: str

# streaming=True not required here — astream_events handles token-level streaming at the graph level
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# think_node adds reasoning context before answering — remove it if latency matters more than answer quality
def think_node(state):
    r = llm.invoke([SystemMessage(content="Think briefly about the question in 1-2 sentences."), HumanMessage(content=state["question"])])
    return {"answer": r.content}

def answer_node(state):
    r = llm.invoke([SystemMessage(content="Give a final concise answer."), HumanMessage(content=state["question"])])
    return {"answer": r.content}

g = StateGraph(ThinkAnswerState)
g.add_node("think_node",  think_node)
g.add_node("answer_node", answer_node)
# linear chain: think → answer → END; branch here for parallel reasoning paths if you need multiple perspectives
g.add_edge(START, "think_node"); g.add_edge("think_node", "answer_node"); g.add_edge("answer_node", END)
app = g.compile()

# same question for both modes — makes latency comparison apples-to-apples
DEMO_QUESTION = "Explain how photosynthesis works in 2 sentences."

# batch invoke blocks until the full graph finishes — user sees nothing until the last node completes
print("=== Batch invoke (waits for full answer) ===")
t0 = time.time()
r = app.invoke({"question": DEMO_QUESTION, "answer": ""})
print(f"Answer ({time.time()-t0:.1f}s): {r['answer']}")

print("\n=== astream_events (tokens arrive as generated) ===")
# async generator: each token from on_chat_model_stream fires as soon as the model emits it — no buffering
async def stream_demo():
    t0 = time.time()
    token_count = 0
    async for event in app.astream_events({"question": DEMO_QUESTION, "answer": ""}, version="v1"):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                print(chunk, end="", flush=True)
                token_count += 1
    print(f"\n({time.time()-t0:.1f}s, {token_count} tokens streamed)")

asyncio.run(stream_demo())

print("\n=== FastAPI SSE reference (not run in Colab) ===")
print("""
from fastapi import FastAPI
from sse_starlette.sse import EventSourceResponse

api = FastAPI()

@api.get("/stream")
async def stream(question: str):
    async def generate():
        async for event in app.astream_events({"question": question, "answer": ""}, version="v1"):
            if event["event"] == "on_chat_model_stream":
                chunk = event["data"]["chunk"].content
                if chunk:
                    yield {"data": chunk, "event": "token"}
        yield {"data": "[DONE]", "event": "done"}
    return EventSourceResponse(generate())
# uvicorn main:api --host 0.0.0.0 --port 8000
# curl -N http://localhost:8000/stream?question=Hello
""")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*